# Chapter 13 — Implementation in PyTorch

Code extracted from Chapter 13 of the CS725 guide. Each `%%writefile` cell below creates one module file (as in the chapter's file layout), so the modules can import each other exactly as in the original.

| File | Purpose |
|---|---|
| `data.py` | datasets, augmentation, and an index-returning wrapper |
| `models.py` | ResNet-18 for CIFAR and a tiny CNN scorer |
| `train.py` | one training run: mixed precision, cosine schedule, timing, forgetting |
| `scores.py` | EL2N scores averaged over several scorer runs |
| `prune.py` | keep hard / easy / random, optional class balance and offset |
| `run.py` | the main experiment grid, logged to CSV |
| `analysis.py` | plots, scorer agreement, and the hardest images |

## 13.2 Data

Two copies of the training set are loaded: one with augmentation for training and one without for scoring. The `WithIndex` wrapper returns each image's position in the dataset.

In [ ]:
%%writefile data.py
# data.py -- datasets, transforms, and a wrapper that also returns each image's index
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset

MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
plain_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)]) # no augmentation

class WithIndex(Dataset):
    """Returns (image, label, index) so we always know which image is which."""
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x, y = self.ds[i]
        return x, y, i

def get_cifar(root="./data", name="cifar10"):
    C = torchvision.datasets.CIFAR10 if name == "cifar10" else torchvision.datasets.CIFAR100
    train_aug = C(root, train=True, download=True, transform=train_tf) # for training
    train_plain = C(root, train=True, download=True, transform=plain_tf) # for scoring
    test        = C(root, train=False, download=True, transform=plain_tf)
    return train_aug, train_plain, test

## 13.3 Models

The standard CIFAR ResNet-18 (3×3 first conv, no initial downsampling) plus a tiny CNN scorer.

In [ ]:
%%writefile models.py
# models.py -- ResNet-18 (CIFAR version) and a tiny CNN scorer
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()          # identity: y = x + F(x)
        if stride != 1 or cin != cout:           # match shapes when they change
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                          nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return F.relu(out + self.shortcut(x))

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(64), nn.ReLU())
        layers, cin = [], 64
        for cout, stride in [(64, 1), (128, 2), (256, 2), (512, 2)]:
            layers += [BasicBlock(cin, cout, stride), BasicBlock(cout, cout, 1)]
            cin = cout
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)                        # logits

def tiny_cnn(num_classes=10):
    def block(cin, cout):
        return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                              nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(block(3, 32), block(32, 64), block(64, 128),
                          nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, num_classes))

## 13.4 Training

Subset training, mixed precision, cosine schedule, `cuda.synchronize()`-wrapped timing, and free forgetting-event tracking.

In [ ]:
%%writefile train.py
# train.py -- one training run, with timing and optional forgetting tracking
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from data import WithIndex

NUM_WORKERS = 2

def amp(device):
    """Mixed precision (16-bit) on the GPU; plain 32-bit on the CPU."""
    return torch.autocast(device_type=device, dtype=torch.float16,
                          enabled=(device == "cuda"))

def sync(device):
    if device == "cuda":
        torch.cuda.synchronize()   # GPU work is asynchronous: wait before reading the clock

@torch.no_grad()
def evaluate(model, test_ds, device="cuda", bs=512):
    model.eval()
    correct = 0
    for x, y in DataLoader(test_ds, batch_size=bs, num_workers=NUM_WORKERS):
        x, y = x.to(device), y.to(device)
        with amp(device):
            out = model(x)
        correct += (out.argmax(1) == y).sum().item()
    return correct / len(test_ds)

def train(model, train_ds, test_ds, epochs, indices=None, lr=0.1, bs=128,
          seed=0, device="cuda", track_forgetting=False):
    torch.manual_seed(seed)
    np.random.seed(seed)
    ds = WithIndex(train_ds)
    if indices is not None:                     # train on the pruned subset only
        ds = Subset(ds, indices)
    loader = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=(device == "cuda"))
    model = model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(loader))
    scaler = torch.amp.GradScaler(device, enabled=(device == "cuda"))

    N = len(train_ds)
    prev = np.zeros(N, dtype=bool)              # was the example correct last time?
    forgets = np.zeros(N, dtype=np.int32)       # number of forgetting events
    learned = np.zeros(N, dtype=bool)           # ever classified correctly?

    sync(device); t0 = time.time()
    for ep in range(epochs):
        model.train()
        for x, y, idx in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with amp(device):                      # mixed precision
                out = model(x)
                loss = F.cross_entropy(out, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            if track_forgetting:
                correct = (out.argmax(1) == y).cpu().numpy()
                idx = idx.numpy()
                forgets[idx] += prev[idx] & ~correct # a 1 -> 0 transition
                prev[idx] = correct
                learned[idx] |= correct
    sync(device); train_time = time.time() - t0

    acc = evaluate(model, test_ds, device)
    extra = {}
    if track_forgetting:
        f = forgets.astype(float)
        f[~learned] = f.max() + 1               # never learned = hardest of all
        extra["forgetting"] = f
    return model, acc, train_time, extra

## 13.5 Scores

EL2N = ‖p − y‖₂ per training image, averaged over several scorer runs. Returns the total scoring cost.

In [ ]:
%%writefile scores.py
# scores.py -- EL2N scores, averaged over several independently trained scorers
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from train import train, sync, amp, NUM_WORKERS

@torch.no_grad()
def el2n_scores(model, train_plain, num_classes, device="cuda", bs=512):
    model.eval()                                  # use stored BatchNorm statistics
    out = []
    loader = DataLoader(train_plain, batch_size=bs, shuffle=False, # keep order!
                        num_workers=NUM_WORKERS)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with amp(device):
            logits = model(x)
        p = F.softmax(logits.float(), dim=1)
        err = p - F.one_hot(y, num_classes).float()       # the error vector p - y
        out.append(err.norm(dim=1).cpu())                 # its length = EL2N
    return torch.cat(out).numpy()

def compute_el2n(make_model, train_aug, train_plain, test, num_classes,
                 epochs, runs, device="cuda"):
    """Train `runs` scorers for `epochs` epochs each; return mean EL2N and total cost."""
    all_scores, total_time = [], 0.0
    for r in range(runs):
        model, acc, t_train, _ = train(make_model(), train_aug, test, epochs,
                                       seed=1000 + r, device=device)
        sync(device); t0 = time.time()
        s = el2n_scores(model, train_plain, num_classes, device)
        sync(device); total_time += t_train + (time.time() - t0)
        all_scores.append(s)
        print(f"scorer run {r}: test acc {acc:.3f}, {t_train:.0f}s")
    return np.mean(all_scores, axis=0), total_time

## 13.6 Pruning

`select` returns the indices to keep (hard / easy / random), optionally class-balanced, with an optional offset to skip the very hardest examples.

In [ ]:
%%writefile prune.py
# prune.py -- choose which examples to keep
import numpy as np

def select(scores, f, strategy, labels=None, seed=0, class_balanced=False, offset=0.0):
    """Return sorted indices of the examples to keep.
    scores: array of length N, larger = harder.   f: keep fraction in (0, 1].
    strategy: 'hard', 'easy' or 'random'. offset: fraction of the very hardest to skip."""
    rng = np.random.default_rng(seed)

    def pick(idx):
        n = int(round(f * len(idx)))
        if strategy == "random":
            return rng.choice(idx, n, replace=False)
        order = idx[np.argsort(-scores[idx], kind="stable")]   # hardest first
        if strategy == "hard":
            skip = int(round(offset * len(idx)))
            return order[skip:skip + n]
        if strategy == "easy":
            return order[::-1][:n]
        raise ValueError(strategy)

    if class_balanced:                     # keep the same fraction of every class
        labels = np.asarray(labels)
        kept = [pick(np.where(labels == c)[0]) for c in np.unique(labels)]
        return np.sort(np.concatenate(kept))
    return np.sort(pick(np.arange(len(scores))))

## 13.7 The experiment grid

Scores once, saves to `.npy`, then runs every (fraction, strategy, seed) combination and appends one row per run to `results.csv`.

In [ ]:
%%writefile run.py
# run.py -- the main pruning grid, logging every run to a CSV file
import csv, os
import numpy as np
from data import get_cifar
from models import ResNet18, tiny_cnn
from scores import compute_el2n
from prune import select
from train import train

def log_row(path, **row):
    new = not os.path.exists(path)
    with open(path, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(row))
        if new:
            w.writeheader()
        w.writerow(row)

if __name__ == "__main__":
    DEVICE, EPOCHS, K = "cuda", 60, 10
    train_aug, train_plain, test = get_cifar(name="cifar10")
    labels = np.array(train_plain.targets)

    # 1) scores from a cheap scorer (saved once, reused by everyone)
    if not os.path.exists("el2n_tiny_e5.npy"):
        s, cost = compute_el2n(lambda: tiny_cnn(K), train_aug, train_plain, test, K,
                               epochs=5, runs=3, device=DEVICE)
        np.save("el2n_tiny_e5.npy", s)
        log_row("scoring_costs.csv", scorer="tiny", epochs=5, runs=3,
                seconds=round(cost, 1))
    scores = np.load("el2n_tiny_e5.npy")

    # 2) the grid: keep fraction x strategy x seed
    for f in [0.9, 0.7, 0.5, 0.3]:
        for strategy in ["hard", "easy", "random"]:
            for seed in [0, 1, 2]:
                idx = select(scores, f, strategy, labels=labels, seed=seed)
                _, acc, t, _ = train(ResNet18(K), train_aug, test, EPOCHS,
                                     indices=idx, seed=seed, device=DEVICE)
                log_row("results.csv", model="resnet18", scorer="tiny_e5",
                        strategy=strategy, fraction=f, seed=seed, n_kept=len(idx),
                        test_acc=round(acc, 4), train_seconds=round(t, 1))

## 13.8 Analysis

`plot_grid`, `agreement`, and `show_hardest`.

In [ ]:
%%writefile analysis.py
# analysis.py -- plots, scorer agreement, and a look at the hardest images
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

def plot_grid(csv="results.csv", out="acc_vs_fraction.png"):
    df = pd.read_csv(csv)
    g = (df.groupby(["strategy", "fraction"])["test_acc"]
           .agg(["mean", "std", "count"]).reset_index())
    for strategy, sub in g.groupby("strategy"):
        plt.errorbar(sub["fraction"] * 100, sub["mean"] * 100, yerr=sub["std"] * 100,
                     marker="o", capsize=3, label=strategy)
    plt.xlabel("% of training data kept"); plt.ylabel("test accuracy (%)")
    plt.legend(); plt.tight_layout(); plt.savefig(out, dpi=200); plt.close()
    return g

def overlap(a, b, f):
    """Fraction of the n hardest under score a that are also among the n hardest under b."""
    n = int(f * len(a))
    A = set(np.argsort(-a)[:n]); B = set(np.argsort(-b)[:n])
    return len(A & B) / n

def agreement(a, b, fractions=(0.9, 0.7, 0.5, 0.3)):
    rho = spearmanr(a, b).correlation
    print(f"Spearman rho = {rho:.3f}")
    for f in fractions:   # compare with the random-overlap baseline f
        print(f"keep {f:.0%}: overlap {overlap(a, b, f):.3f} (random: {f:.2f})")

def show_hardest(scores, k=20, out="hardest.png", root="./data"):
    import torch, torchvision, torchvision.transforms as T, torchvision.utils as vu
    raw = torchvision.datasets.CIFAR10(root, train=True, download=True)
    idx = np.argsort(-scores)[:k]
    imgs = torch.stack([T.ToTensor()(raw[i][0]) for i in idx])
    vu.save_image(imgs, out, nrow=10)
    print("labels:", [raw.classes[raw.targets[i]] for i in idx])

## Running it

Enable a GPU first (Kaggle: Settings → Accelerator; Colab: Runtime → Change runtime type). `run.py` guards its main block with `if __name__ == "__main__"`, so run it as a script:

In [ ]:
!python run.py

After the grid finishes, use the analysis helpers:

In [ ]:
from analysis import plot_grid, agreement, show_hardest
import numpy as np

print(plot_grid())                       # writes acc_vs_fraction.png
scores = np.load("el2n_tiny_e5.npy")
show_hardest(scores)                     # writes hardest.png

## 13.9 Working on Kaggle or Colab (notes from the chapter)

- Turn on the GPU and record it with `torch.cuda.get_device_name()`.
- Save to persistent storage (Kaggle: `/kaggle/working/`; Colab: mount Google Drive). Sessions end without warning.
- Share the `.npy` score files so nobody recomputes them.
- Do a quick test first (e.g. 2 epochs, 1 seed) before spending hours.
- Free GPU quotas are limited; split the grid by seed or by keep fraction across accounts.
- `num_workers=2` is a good default on both platforms.